# Clasificacion de Fatiga en Ciclismo - LSTM con Atencion
# Luis Nieto - Gabriela Osorio

Este notebook entrena un modelo LSTM con mecanismo de atencion para clasificar
sesiones de ciclismo como pedaleo normal o Fatiga.

Va a analizar secuencias de movimiento de 4 ángulos articulares (rodilla, cadera, tronco y tobillo)
Se usa red neuronal tipo LSTM (memoria de largo plazo) con mecanismo de atención
Clasifica cada sesión en 2 estados (0 es normal; 1 es fatiga)

La validación se hace con LOSO (Leave one subject Out)
* Entrena con 4 ciclista y prueba con el 5to, rotando hasta probar con todos

## 0. Importaciones y configuracion

In [ ]:
#Importamos las librerias necesarias
import os
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, filtfilt

#librerias de pytorch para crear y entrenar la red neuronal
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

#Configuración para los gráficos
plt.rcParams.update({
    'figure.facecolor': '#0a0f14',
    'axes.facecolor': '#0d1520',
    'axes.edgecolor': '#1a3a5c',
    'axes.labelcolor': '#8892A4',
    'axes.titlecolor': '#F0F2F5',
    'xtick.color': '#8892A4',
    'ytick.color': '#8892A4',
    'grid.color': '#1a3a5c',
    'grid.alpha':  0.4,
    'text.color': '#F0F2F5',
    'legend.facecolor': '#0d1520',
    'legend.edgecolor': '#1a3a5c',
    'font.family': 'monospace',
})

#Colores para las condiciones
COLOR_INICIAL = '#00a3e0'
COLOR_FATIGA  = '#78be20'

# Detectar dispositivo: MPS (Apple M4), CUDA, o CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Usando GPU Apple Silicon (MPS)')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('Usando GPU CUDA')
else:
    DEVICE = torch.device('cpu')
    print('Usando CPU')

#Parámetros del experimento
FPS  = 29.97
VENTANA  = 90 # 90 frames = 3 segundos
OVERLAP = 45 #Solapamiento del 50% entre ventanas
ANGULOS = ['Rodilla_I', 'Cadera_I', 'Tronco', 'Tobillo_I']
SUJETOS = ['P1', 'P2', 'P3', 'P4', 'P5']
CONDICIONES = ['Inicial', 'Fatiga']
BASE = Path('..')
OUTPUTS = BASE / 'outputs'

#Fijamos semillas para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

print(f'Ventana: {VENTANA} frames ({VENTANA/FPS:.1f} segundos)')
print(f'Overlap: {OVERLAP} frames ({OVERLAP/FPS:.1f} segundos)')
print(f'Angulos: {ANGULOS}')

## 1. Funciones de procesamiento

In [ ]:
# Se replican las mismas funciones del notebook de analisis exploratorio para que este notebook sea autocontenido.
def extraer_punto(keypoints, indice):
    # Extrae las coordenadas (x, y, confianza) de un keypoint específico
    b = indice * 3
    return (keypoints[b], keypoints[b+1], keypoints[b+2])


def calcular_angulo(p1, p2, p3, umbral=0.1):
    # Calcula el ángulo en p2 formado por los puntos p1-p2-p3
    # Retorna None si algún punto tiene baja confianza
    if any(p[2] < umbral for p in [p1, p2, p3]):
        return None
    v1 = (p1[0]-p2[0], p1[1]-p2[1])
    v2 = (p3[0]-p2[0], p3[1]-p2[1])
    dot = v1[0]*v2[0] + v1[1]*v2[1]
    m1 = math.sqrt(v1[0]**2 + v1[1]**2)
    m2 = math.sqrt(v2[0]**2 + v2[1]**2)
    if m1 == 0 or m2 == 0:
        return None
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot/(m1*m2))))), 2)


def calcular_angulo_tronco(hombro, cadera, umbral=0.1):
    # Calcula la inclinación del tronco respecto a la vertical
    if hombro[2] < umbral or cadera[2] < umbral:
        return None
    vx = hombro[0] - cadera[0]
    vy = hombro[1] - cadera[1]
    mag = math.sqrt(vx**2 + vy**2)
    if mag == 0:
        return None
    dot = (vx*0 + vy*(-1)) / mag
    return round(math.degrees(math.acos(max(-1.0, min(1.0, dot)))), 2)


def seleccionar_ciclista(personas, ancho_frame=1920):
    # Cuando hay múltiples personas, selecciona la más cercana al centro
    if len(personas) == 1:
        return personas[0]['pose_keypoints_2d']
    centro_frame = ancho_frame / 2
    mejor_idx, menor_dist = 0, float('inf')
    for idx, persona in enumerate(personas):
        kp = persona['pose_keypoints_2d']
        # Usamos puntos del torso para calcular el centro de la persona
        puntos_x = [kp[i*3] for i in [1,2,5,8,9,12]
                    if kp[i*3+2] > 0.1 and kp[i*3] > 0]
        if not puntos_x:
            continue
        dist = abs(sum(puntos_x)/len(puntos_x) - centro_frame)
        if dist < menor_dist:
            menor_dist = dist
            mejor_idx  = idx
    return personas[mejor_idx]['pose_keypoints_2d']


def butter_lowpass(serie, fc=6.0, fps=29.97, orden=4):
    # Aplica filtro Butterworth para suavizar la señal (elimina ruido)
    nyquist = fps / 2.0
    b, a = butter(orden, fc/nyquist, btype='low', analog=False)
    serie_interp = serie.interpolate(method='linear', limit_direction='both')
    nan_mask = serie.isna()
    if serie_interp.notna().sum() < 15:
        return serie.values
    filtrada = filtfilt(b, a, serie_interp.values)
    filtrada[nan_mask] = np.nan
    return filtrada

## 2. Carga y preprocesamiento de datos

Se cargan los JSON, se calculan los angulos, se aplica filtro Butterworth
y se normaliza con Z-score por angulo.

In [ ]:
def cargar_serie(sujeto, condicion, outputs_dir, fps=29.97):
    # Carga los JSON de un sujeto/condicion.
    # Retorna DataFrame con 4 angulos por frame, ya filtrados.
    json_dir = outputs_dir / f'{sujeto}_{condicion}' / 'json'
    archivos = sorted([f for f in os.listdir(json_dir)
                       if f.endswith('_keypoints.json')])
    #Detectamos el ancho del frame desde los primeros JSON
    ancho_frame = 1920
    for nombre in archivos[:10]:
        try:
            with open(json_dir / nombre) as f:
                d = json.load(f)
            if d.get('people'):
                kp = d['people'][0]['pose_keypoints_2d']
                xs = [kp[i*3] for i in range(25) if kp[i*3] > 0]
                if xs:
                    ancho_frame = max(xs) * 1.1
                    break
        except Exception:
            continue
    #Procesamos cada frame del video
    filas = []
    for nombre in archivos:
        try:
            with open(json_dir / nombre) as f:
                datos = json.load(f)
        except Exception:
            continue
        #Si no se detectó ninguna persona en este frame
        if not datos.get('people'):
            filas.append({'Rodilla_I': None, 'Cadera_I': None,
                          'Tronco': None, 'Tobillo_I': None})
            continue
        #Seleccionamos el ciclista principal y extraemos sus puntos clave
        kp = seleccionar_ciclista(datos['people'], ancho_frame)
        p = {n: extraer_punto(kp, n) for n in range(25)}
        #Calculamos los ángulos de interés y los guardamos en la lista de filas
        filas.append({
            'Rodilla_I': calcular_angulo(p[12], p[13], p[14]),
            'Cadera_I': calcular_angulo(p[1],  p[12], p[13]),
            'Tronco': calcular_angulo_tronco(p[5], p[12]),
            'Tobillo_I': calcular_angulo(p[13], p[14], p[19]),
        })

    #Creamos el dataframe y aplicamos el filtro a cada ángulo
    df = pd.DataFrame(filas)
    for angulo in ANGULOS:
        df[angulo] = butter_lowpass(df[angulo], fps=fps)
    return df

#Cargamos todos los sujetos y condiciones
datos_crudos = {}
for sujeto in SUJETOS:
    for condicion in CONDICIONES:
        key = f'{sujeto}_{condicion}'
        df  = cargar_serie(sujeto, condicion, OUTPUTS, FPS)
        datos_crudos[key] = df
        frames_validos = df[ANGULOS].dropna().shape[0]
        print(f'  {key}: {len(df)} frames, {frames_validos} validos')


In [ ]:
# Normalizacion Z-score con estadisticas globales para que todos los ángulos
#estén en la misma escala

#Concatenamos los dataframes para calcular estadísticas globales
todos = pd.concat(list(datos_crudos.values()), ignore_index=True)

#Calculamos la media y desv. estánfar de cada ángulo
MEDIA_GLOBAL = {}
STD_GLOBAL = {}
for angulo in ANGULOS:
    MEDIA_GLOBAL[angulo] = todos[angulo].dropna().mean()
    STD_GLOBAL[angulo] = todos[angulo].dropna().std()
    print(f'{angulo}: media={MEDIA_GLOBAL[angulo]:.2f}, std={STD_GLOBAL[angulo]:.2f}')

#Normalizamos (valor- media)/ desv. estándar 
# Ontendremos media=0 y std =1
def normalizar(df):
    df_norm = df.copy()
    for angulo in ANGULOS:
        df_norm[angulo] = (df[angulo] - MEDIA_GLOBAL[angulo]) / STD_GLOBAL[angulo]
        df_norm[angulo] = df_norm[angulo].fillna(0.0)
    return df_norm

#Aplicamos la normalización a todos los datos
datos_norm = {key: normalizar(df) for key, df in datos_crudos.items()}

## 3. Ventanas deslizantes

Cada video se divide en ventanas de 90 frames con 45 de overlap (50%).
Cada ventana hereda la etiqueta del video completo:
- Inicial = 0
- Fatiga  = 1

In [ ]:
def generar_ventanas(df_norm, etiqueta, sujeto, ventana=90, overlap=45):
    #Divide una serie temporal en ventanas deslizantes.
    # Solo incluye ventanas con al menos 70% de frames validos.

    valores = df_norm[ANGULOS].values
    paso = ventana - overlap #Frames que avanzamos en cada paso
    muestras = []

    #Recorremos la serie en ventanas deslizantes
    for inicio in range(0, len(valores) - ventana + 1, paso):
        fragmento = valores[inicio: inicio + ventana]
        #Solo guardamos la ventana si tiene menos del 30% de frames faltantes
        if np.isnan(fragmento).mean() < 0.3:
            fragmento = np.nan_to_num(fragmento, nan=0.0)
            muestras.append({
                'X': fragmento.astype(np.float32), #Secuencia de entrada
                'y': etiqueta, #Clase: 0=Inicial, 1=Fatiga
                'sujeto': sujeto,
            })
    return muestras

# Generamos todas las ventanas de todos los sujetos y condiciones
todas_las_muestras = []
print(f'Ventana={VENTANA} frames, overlap={OVERLAP} frames, paso={VENTANA-OVERLAP} frames')

for sujeto in SUJETOS:
    for condicion, etiqueta in zip(CONDICIONES, [0, 1]):
        key = f'{sujeto}_{condicion}'
        muestras = generar_ventanas(datos_norm[key], etiqueta, sujeto, VENTANA, OVERLAP)
        todas_las_muestras.extend(muestras)
        print(f'  {key}: {len(muestras)} ventanas (etiqueta={etiqueta})')


# Mostramos el balance de clases
total = len(todas_las_muestras)
n_inicial = sum(1 for m in todas_las_muestras if m['y'] == 0)
n_fatiga  = sum(1 for m in todas_las_muestras if m['y'] == 1)
print(f'\nTotal: {total} | Inicial: {n_inicial} | Fatiga: {n_fatiga}')

## 4. Dataset y DataLoader de PyTorch

In [ ]:
class CiclistaDataset(Dataset):
    #clase para manejar nuestros datos en Pytorch
    #Cada muestra es una ventana de 90 frames y 4 ángulos
    def __init__(self, muestras):
        #Guardamos la lista de ventana que generamos antes
        self.muestras = muestras

    def __len__(self):
        #Retorna cuantas ventana tenemos en total
        return len(self.muestras)

    def __getitem__(self, idx):
        #Obtiene una ventana específica y la convierte a tensores de PyTorch
        m = self.muestras[idx]
        X = torch.tensor(m['X'], dtype=torch.float32) #Secuencia de angulo
        y = torch.tensor(m['y'], dtype=torch.float32) #Etiqueta de 0 o 1 
        return X, y

print(f'Shape de cada muestra X: ({VENTANA}, {len(ANGULOS)})')

## 5. Arquitectura LSTM con Atencion

La arquitectura tiene tres componentes:

1. **LSTM**: aprende patrones temporales en la secuencia de movimientos
2. **Capa de atencion**: identifica que frames son más importantes para la clasificación
3. **Clasificador**: decide si hay fatiga o no basándose en la información relevante

In [ ]:
#Con atención
class CapaAtencion(nn.Module):
    #Mecanismo de atención que asigna importancia a cada frame de la secuencia

    #Los frames más relevantes reciben un mayor peso en la decisión final
    def __init__(self, hidden_size):
        super().__init__()
        self.atencion = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, salidas_lstm):
        # salidas_lstm: (batch, ventana, hidden_size)

        #Score de importamcia por frame
        scores = self.atencion(salidas_lstm)
        #Normalizamos los scores con softmax para obtener pesos de importancia
        pesos = torch.softmax(scores, dim=1)
        #Creamos el vector de contexto como suma ponderada de las salidas del LSTM
        contexto = (pesos * salidas_lstm).sum(dim=1)
        return contexto, pesos.squeeze(-1) # pesos: (batch, ventana)


class LSTMAtencion(nn.Module):
    #Modelo completo para clasificar fatiga usando LSTM con atención
    #Recibe secuencias de angulos y predice 0 o 1 (inicial o fatiga)
    def __init__(self, n_angulos=4, hidden_size=32, n_capas=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_angulos, #4 angulos de entrada
            hidden_size=hidden_size, #32 neuronas (evitar overfitting)
            num_layers=n_capas, #2 capaas apiladas 
            batch_first=True,
            dropout=dropout if n_capas > 1 else 0.0,  #Regularización
        )
        self.atencion = CapaAtencion(hidden_size) #Mecanismo de atención
        self.dropout = nn.Dropout(dropout) # Más Regularización
        self.clasificador = nn.Linear(hidden_size, 1) #Decisión final

    def forward(self, x):
        # x: (batch, ventana, n_angulos)

        # LSTM procesa toda la secuencia 
        salidas, _  = self.lstm(x) 
        #Atención selecciona lo más importante
        contexto, pesos = self.atencion(salidas)
        #Dropout para evitar sobreajuste
        contexto = self.dropout(contexto)
        # Clasificador final que predice la probabilidad de fatiga
        logit = self.clasificador(contexto).squeeze(-1)
        return logit, pesos


# Sin atención
class LSTMSinAtencion(nn.Module):
    # Usa el último estado oculto del LSTM para clasificar.
    def __init__(self, n_angulos=4, hidden_size=32, n_capas=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_angulos,
            hidden_size=hidden_size,
            num_layers=n_capas,
            batch_first=True,
            dropout=dropout if n_capas > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.clasificador = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, ventana, n_angulos)
        salidas, (h_n, c_n) = self.lstm(x)
        
        # Tomamos el último estado oculto de la última capa
        # h_n: (n_capas, batch, hidden_size)
        ultimo_estado = h_n[-1]  # (batch, hidden_size)
        
        # Dropout y clasificación
        ultimo_estado = self.dropout(ultimo_estado)
        logit = self.clasificador(ultimo_estado).squeeze(-1)
        
        # Retornamos None para pesos (compatibilidad con el código existente)
        return logit, None

In [ ]:
## 5.5. Selección de arquitectura

USAR_ATENCION = False  # True = con atención, False = sin atención

if USAR_ATENCION:
    print('Entrenando CON atención')
    ModeloAUsar = LSTMAtencion
else:
    print('Entrenando SIN atención')
    ModeloAUsar = LSTMSinAtencion

## 6. Entrenamiento con Leave-One-Subject-Out (LOSO)

En cada iteración usamos un sujeto como prueba y los 4 restantes para entrenar
Esta estrategia es estándar cuando hay pocos sujetos: evalúa si el modelo
puede generalizar a personas nuevas que nunca ha visto

In [ ]:
def entrenar_epoca(modelo, loader, optimizer, criterio, device):
    #Entrena el modelo durante una epoca completa
    # Retorna la pérdida promedio de todos los batches

    modelo.train() #Activamos el modo entrenamiento
    perdida_total = 0.0

    for X_batch, y_batch in loader:
        #Movemos los datos al dispositivo (GPU o CPU)
        X_batch = X_batch.to(device) 
        y_batch = y_batch.to(device)

        #Reiniciamos los gradientes acumulados
        optimizer.zero_grad()

        # Forward pass: obtenemos las predicciones y calculamos la pérdida
        logit, _ = modelo(X_batch)
        perdida = criterio(logit, y_batch) #Que tan equivocada está la predicción
        perdida.backward()
        # Gradient clipping para estabilidad del entrenamiento
        nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        
        #Actualizamos los pesos del modleo
        optimizer.step()
        perdida_total += perdida.item()
    return perdida_total / len(loader)


def evaluar(modelo, loader, criterio, device):
    #Evalúa el modelo en el conjunto de validación o prueba
    # Retorna la pérdida promedio y la accuracy
    modelo.eval() #Modo de evaluación 
    perdida_total = 0.0
    correctos = 0
    total = 0
    #No calculamos gradiente durante la evaluación para ahorrar memoria y acelerar el proceso
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            #Calculamos la predicción
            logit, _ = modelo(X_batch)
            #Calculamos la pérdida
            perdida = criterio(logit, y_batch)
            perdida_total += perdida.item()
            #convertimos el logit a probabilidad con sigmoid y luego a clase binaria con umbral 0.5
            preds = (torch.sigmoid(logit) > 0.5).float()
            #contamos cuantas predicciones fueorn correctas
            correctos += (preds == y_batch).sum().item()
            total += len(y_batch)
    acc = correctos / total if total > 0 else 0.0
    return perdida_total / len(loader), acc

In [ ]:
# Hiperparametros
# Arquitectura pequena intencionalmente: el dataset tiene solo 10 sesiones (5 sujetos x 2 condiciones)
# Una red mas grande sobreajustaria los datos de entrenamiento


HIDDEN_SIZE = 32 #Neuronas ocultas
N_CAPAS = 2 #Capas apiladas
DROPOUT = 0.3 # Regularización
LR = 1e-3 #Learning Rate
EPOCAS = 80 #Numero de veces que el modelo verá todos los datos
BATCH_SIZE = 16 #Ventanas procesadas simultáneamente

resultados_loso = []
historiales = [] #Se guarda el proceso de train

#LOSO: Cada sujeto es el conjunto de prueba una vez, y el resto se usa para entrenamiento
for sujeto_test in SUJETOS:
    print(f'\nFold: sujeto de prueba = {sujeto_test}')

    #Separamos las muestras de entrenamiento y prueba según el sujeto
    train_muestras = [m for m in todas_las_muestras if m['sujeto'] != sujeto_test]
    test_muestras  = [m for m in todas_las_muestras if m['sujeto'] == sujeto_test]

    print(f'Entrenamiento:{len(train_muestras)} ventanas')
    print(f'Prueba:{len(test_muestras)} ventanas')

    # Creamos los DataLoaders para cargar los datos en batches
    train_loader = DataLoader(CiclistaDataset(train_muestras),
                              batch_size=BATCH_SIZE, shuffle=True) #Mezclamos para train
    test_loader = DataLoader(CiclistaDataset(test_muestras),
                              batch_size=BATCH_SIZE, shuffle=False) #No mezclamos para test

    #Creamos un modelo nuevo para este fold
    modelo = ModeloAUsar(n_angulos=4, hidden_size=HIDDEN_SIZE,
                     n_capas=N_CAPAS, dropout=DROPOUT).to(DEVICE)
    #Optimizados adam con weight decay (más regularización)
    optimizer = optim.Adam(modelo.parameters(), lr=LR, weight_decay=1e-4)
    #Función de pérdida para clasificación binaria
    criterio = nn.BCEWithLogitsLoss()

    #Sceduler: reduce el learning rate si el modelo deja deja de mejorar
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5)

    #Variables a guardar el progreso
    historial = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    mejor_acc = 0.0

    #Entrenamos durante todas las epocas
    for epoca in range(EPOCAS):
        #Entrenamos durante todas las épocas
        train_loss = entrenar_epoca(modelo, train_loader, optimizer, criterio, DEVICE)
        # Evaluamos en el conjunto de prueba
        test_loss, test_acc = evaluar(modelo, test_loader, criterio, DEVICE)
        #Ajustamos el learning rate 
        scheduler.step(test_loss)

        #Guardamos las métricas
        historial['train_loss'].append(train_loss)
        historial['test_loss'].append(test_loss)
        historial['test_acc'].append(test_acc)

        #Actualizamos el mejor accurracy
        if test_acc > mejor_acc:
            mejor_acc = test_acc
        #Mostramos el progreso cada 20 épocas
        if (epoca + 1) % 20 == 0:
            print(f'Epoca {epoca+1:3d}/{EPOCAS}'
                  f'Loss train: {train_loss:.4f} | '
                  f'Loss test: {test_loss:.4f} | '
                  f'Acc: {test_acc:.3f}')

    # Guardamos los resultados de este fold
    resultados_loso.append({
        'sujeto_test': sujeto_test,
        'mejor_acc': mejor_acc,
        'acc_final': historial['test_acc'][-1],
        'modelo': modelo,
    })
    historiales.append(historial)
    print(f'Mejor accuracy: {mejor_acc:.3f}')

## 7. Resultados del LOSO

In [ ]:
from sklearn.metrics import classification_report

accs = []
for r in resultados_loso:
    print(f'{r["sujeto_test"]:12s}'
          f'Mejor acc: {r["mejor_acc"]:.3f}'
          f'Acc final: {r["acc_final"]:.3f}')
    accs.append(r['mejor_acc'])

print(f'\nAccuracy promedio LOSO: {np.mean(accs):.3f} +/- {np.std(accs):.3f}')

for r in resultados_loso:
    sujeto_test = r['sujeto_test']
    modelo = r['modelo'].to(DEVICE)
    modelo.eval()
    test_muestras = [m for m in todas_las_muestras if m['sujeto'] == sujeto_test]
    test_loader = DataLoader(CiclistaDataset(test_muestras),
                               batch_size=BATCH_SIZE, shuffle=False)
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logit, _ = modelo(X_batch.to(DEVICE))
            preds = (torch.sigmoid(logit) > 0.5).cpu().numpy()
            y_pred.extend(preds.astype(int).tolist())
            y_true.extend(y_batch.numpy().astype(int).tolist())
    print(f'\n  {sujeto_test}:')
    print(classification_report(y_true, y_pred, target_names=['Inicial', 'Fatiga'],zero_division=0))

## 8. Curvas de entrenamiento

Visualizamos como evolucionó el entrenamiento en cada fold
Esto nos ayuda a detectar overfitting (el modelo memoriza en vez de aprender)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Curvas de entrenamiento LOSO, LSTM con Atencion')

#Creamos un gráfico para cada sujeto 
for col, (r, historial) in enumerate(zip(resultados_loso, historiales)):
    sujeto = r['sujeto_test']
    epocas = range(1, EPOCAS + 1)

    #Fila superior: Grafico de pérdida de entrenamiento y prueba
    ax_loss = axes[0, col]
    ax_loss.plot(epocas, historial['train_loss'], color=COLOR_INICIAL,
                 linewidth=1.5, label='Train')
    ax_loss.plot(epocas, historial['test_loss'],  color=COLOR_FATIGA,
                 linewidth=1.5, label='Test')
    ax_loss.set_title(f'Test: {sujeto}')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=8)
    ax_loss.grid(True)

    #Fila inferior: Grafico de accuracy
    ax_acc = axes[1, col]
    ax_acc.plot(epocas, historial['test_acc'], color='#772583', linewidth=1.5)
    #Linea de referencias: 0.5 es lo que obtendría un modelo que adivina al azar
    ax_acc.axhline(0.5, color='#cc2244', linestyle='--',
                   linewidth=1, label='Azar (0.5)')
    
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_xlabel('Epoca')
    ax_acc.set_ylim(0, 1.05)
    ax_acc.legend(fontsize=8)
    ax_acc.grid(True)

plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'lstm_curvas_entrenamiento.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0f14')
plt.show()
print('Figura guardada: outputs/figures/lstm_curvas_entrenamiento.png')

## 9. Interpretabilidad - Pesos de atencion

Los pesos de atencion indican en que momentos de la ventana temporal
el modelo se basa para clasificar. Pesos altos = frames criticos.
Esto permite explicar por que el modelo clasifica una ventana como Inicial o Fatiga.

In [ ]:
if USAR_ATENCION:
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle('Pesos de atencion promedio: Inicial vs Fatiga')

    #Convertimos frames a tiempo en segundos para el eje x
    tiempo_ventana = np.arange(VENTANA) / FPS

    for col, r in enumerate(resultados_loso):
        sujeto_test = r['sujeto_test']
        modelo = r['modelo'].to(DEVICE)
        modelo.eval()
        #Obtenemos las muestras de prueba de este sujeto
        test_muestras = [m for m in todas_las_muestras if m['sujeto'] == sujeto_test]

        #Recolectamos los pesos de atención separados por clase (inicial vs fatiga)
        pesos_inicial, pesos_fatiga = [], []
        with torch.no_grad():
            for muestra in test_muestras:
                #Obtenemos los pesos de atención para esta ventana de prueba
                X = torch.tensor(muestra['X']).unsqueeze(0).to(DEVICE)
                _, pesos = modelo(X)
                pesos_np = pesos.squeeze(0).cpu().numpy()
                #Separamos según la clase
                if muestra['y'] == 0:
                    pesos_inicial.append(pesos_np)
                else:
                    pesos_fatiga.append(pesos_np)

        # Graficamos los pesos promedio de cada clase
        ax = axes[col]
        if pesos_inicial:
            ax.plot(tiempo_ventana, np.mean(pesos_inicial, axis=0),
                    color=COLOR_INICIAL, linewidth=1.5, label='Inicial')
        if pesos_fatiga:
            ax.plot(tiempo_ventana, np.mean(pesos_fatiga, axis=0),
                    color=COLOR_FATIGA, linewidth=1.5, label='Fatiga')
        ax.set_title(f'Test: {sujeto_test}')
        ax.set_xlabel('Tiempo en ventana (s)')
        ax.set_ylabel('Peso de atencion')
        ax.legend(fontsize=8)
        ax.grid(True)

    #Guardamos la figura
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'figures' / 'lstm_pesos_atencion.png',
                dpi=150, bbox_inches='tight', facecolor='#0a0f14')
    plt.show()
    print('Figura guardada: outputs/figures/lstm_pesos_atencion.png')
else: 
    print('No se muestran pesos de atención porque el modelo no los utiliza.')

## 10. Guardar resultados

In [ ]:
## 10. Guardar resultados

# Nombre del archivo según la versión
sufijo = 'con_atencion' if USAR_ATENCION else 'sin_atencion'

# Guardar métricas en Excel
df_resultados = pd.DataFrame([{
    'Sujeto_test': r['sujeto_test'],
    'Mejor_acc':   round(r['mejor_acc'], 4),
    'Acc_final':   round(r['acc_final'], 4),
} for r in resultados_loso])

# Agregamos una fila con el promedio de cada métrica al final del DataFrame
df_resultados.loc[len(df_resultados)] = {
    'Sujeto_test': 'PROMEDIO',
    'Mejor_acc':   round(df_resultados['Mejor_acc'].mean(), 4),
    'Acc_final':   round(df_resultados['Acc_final'].mean(), 4),
}

# Guardar Excel con el sufijo apropiado
df_resultados.to_excel(
    OUTPUTS / 'metrics' / f'lstm_resultados_loso_{sufijo}.xlsx', 
    index=False
)

# Guardar pesos de cada modelo con el sufijo apropiado
for r in resultados_loso:
    ruta = OUTPUTS / 'models' / f'lstm_{r["sujeto_test"]}_{sufijo}.pt'
    torch.save(r['modelo'].state_dict(), ruta)

print('Resultados guardados.')
print(f'Métrica: outputs/metrics/lstm_resultados_loso_{sufijo}.xlsx')
print(f'Modelos: outputs/models/lstm_{{sujeto}}_{sufijo}.pt')
print()
print('Resumen final:')
print(df_resultados.to_string(index=False))
print(f'\nAccuracy promedio LOSO: {np.mean(accs):.3f} +/- {np.std(accs):.3f}')

In [ ]:
## 11. Análisis de Estabilidad (Múltiples Seeds)

print('PROBANDO ESTABILIDAD CON DIFERENTES SEEDS - LSTM')

SEEDS_TO_TEST = [42, 123, 456, 789, 999]
resultados_por_seed_lstm = []

for seed in SEEDS_TO_TEST:
    print(f'Entrenando con seed = {seed}')
    
    # Fijar seeds
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Entrenar LOSO con este seed
    accs_temp = []
    
    for sujeto_test in SUJETOS:
        # Separar datos
        train_muestras = [m for m in todas_las_muestras if m['sujeto'] != sujeto_test]
        test_muestras  = [m for m in todas_las_muestras if m['sujeto'] == sujeto_test]
        
        train_loader = DataLoader(CiclistaDataset(train_muestras),
                                  batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(CiclistaDataset(test_muestras),
                                 batch_size=BATCH_SIZE, shuffle=False)
        
        # Crear modelo
        modelo = ModeloAUsar(n_angulos=4, hidden_size=HIDDEN_SIZE,
                     n_capas=N_CAPAS, dropout=DROPOUT).to(DEVICE)
        optimizer = optim.Adam(modelo.parameters(), lr=LR, weight_decay=1e-4)
        criterio = nn.BCEWithLogitsLoss()
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=10, factor=0.5)
        
        # Entrenar
        mejor_acc = 0.0
        for epoca in range(EPOCAS):
            train_loss = entrenar_epoca(modelo, train_loader, optimizer, criterio, DEVICE)
            test_loss, test_acc = evaluar(modelo, test_loader, criterio, DEVICE)
            scheduler.step(test_loss)
            
            if test_acc > mejor_acc:
                mejor_acc = test_acc
        
        accs_temp.append(mejor_acc)
        print(f'{sujeto_test}: {mejor_acc:.4f}')
    
    # Guardar resultados de este seed
    resultados_por_seed_lstm.append({
        'seed': seed,
        'accuracies': accs_temp.copy(),
        'mean': np.mean(accs_temp),
        'std': np.std(accs_temp)
    })
    print(f'\nPromedio con seed {seed}: {np.mean(accs_temp):.4f} ± {np.std(accs_temp):.4f}')

print('RESUMEN DE ESTABILIDAD - LSTM')

# Tabla comparativa
df_stability_lstm = pd.DataFrame(resultados_por_seed_lstm)
print('\nResultados por seed:')
print(df_stability_lstm[['seed', 'mean', 'std']].to_string(index=False))

# Variabilidad entre seeds
means = df_stability_lstm['mean'].values
print(f'\n\nVariabilidad entre seeds:')
print(f'Mean accuracy: {np.mean(means):.4f}')
print(f'Std entre seeds: {np.std(means):.4f}')
print(f'Rango: [{np.min(means):.4f}, {np.max(means):.4f}]')

# Guardar
df_stability_lstm.to_excel(OUTPUTS / 'metrics' / 'lstm_stability_analysis.xlsx', 
                            index=False)
print('\n  Análisis guardado en: outputs/metrics/lstm_stability_analysis.xlsx')